## Notebook 1 

### On this notebook we are going to evaluate the missing values from the previous webscrapping

Context:
<p>After the webscrapping of the worldfootball.net website, some players were found with missing data.</p>
<p>This Notebook main goal is to account these players.</p>

<p>Goals:</p>
<p>***Create a dataframe with the name of all players with missing data from all worldcups</p>
<p>***Create a dataframe with the frequence and the percentage of missing data from each worldcup</p>

In [1]:
import numpy as np
import pandas as pd

In [2]:
def list_of_countries(df):
    # df - DataFrame with the data from a worldcup
    # countries - list with name of the countries in the df
    tuplas = [tupla for i,tupla in enumerate(df.columns) if i%5 ==0]
    countries = [tupla[0] for tupla in tuplas]
    return countries

Functions for create the dataframe with the name of players without date of birth:

In [3]:
def concat_country_w_player(country,missing_players,missings):
    #Place each player with the country for which he played in the worldcup
    #Return a DataFrame with the structure country-player
    country_played = np.repeat(country,len(missing_players))
    missing_players_wcountry = np.vstack((country_played,missing_players)) #Place the name of the country of each player together
    missing_players_wcountry = pd.DataFrame(missing_players_wcountry.T)
    missings = pd.concat([missings,missing_players_wcountry])
    return missings


In [4]:
def players_without_data(df):
    # missings - DataFrame with the name of the players who don't have date of birth and the country of them
    countries = list_of_countries(df)
    missings = pd.DataFrame([])
    for country in countries:
        dfc = df[country]['Day']
        try:
            filtering = dfc[dfc == 0]
            filtering = filtering.index
            filtering2 = df[country]['Name'][filtering] != '0'
        except:
            print('There is no missing in {}'.format(country)) # Verification
            continue
        missing_players = df[country]['Name'][filtering][filtering2] #List with the name of players with missing data
        
        if(len(missing_players != 0)):
            missings = concat_country_w_player(country,missing_players,missings)
        else:
            continue
            
    if(len(missings != 0)): # If exist missing data:
        missings.columns = ['Country','Players without data']
        return missings
    else: # If doesn't exist missing data:
        return [0]
        

In [5]:
def missings_by_team(df):
    #new_df - DataFrame in which the players of interest are organized by team as "players_without_data" does, 
    #but the columns names are entitled with the name of the team 
    missings = players_without_data(df)
    missings.set_index('Country',drop = True,inplace = True)
    teams = missings.index.unique()
    new_df = pd.DataFrame([],columns = teams )
    for team in teams:
        n_missings = missings.loc[team].shape[0]
        len_df = new_df.shape[0]
        if(n_missings == 1):
            
            if(len_df == 0):
                to_complete_row = np.zeros(new_df.shape[1],dtype=int)
                new_df = pd.DataFrame(np.vstack([new_df,to_complete_row]),columns = new_df.columns )
                
                
            name_list = missings.loc[team]['Players without data']
            new_df[team][0] = name_list
            continue
            
        elif(n_missings >1):
            name_list = missings.loc[team]['Players without data'].values
            if(len_df > len(name_list)):
                to_complete = len_df - len(name_list)
                for i in range(to_complete):
                    name_list = np.append(name_list,0)  
            elif(len_df < len(name_list)):
                to_complete = len(name_list) - len_df
                for i in range(to_complete):
                    to_complete_row = pd.Series(np.zeros(len(teams),dtype = int), index = teams)
                    new_df = pd.DataFrame(np.vstack([new_df,to_complete_row]),columns = teams)        

        new_df[team] = name_list
    return new_df

Functions to create the dataframe with the frequence of players with missing data

In [6]:
def list_countries_with_missings(df):
    # return a list with the countries with missings
    return players_without_data(df)['Country'].unique()
    

In [7]:
def count_missings(df):
    missings = players_without_data(df)
    count_missings = missings.groupby(missings['Country']).count()
    return count_missings

In [8]:
def count_players(df):
    # df - DataFrame of the worldcup
    # df2 - DataFrame with the total players in countries with missing data
    
    missings = players_without_data(df)
    countries_with_missings = list_countries_with_missings(df)
    
    #Calculating the total number of players in these countries:
    tot_players_list = []
    for country in countries_with_missings:
        players = (df[country]['Name'] != '0')
        total_players = (players).value_counts()[0]
        tot_players_list.append(total_players)
    
    #Creating a dataframe :
    df2 = pd.DataFrame([],index = countries_with_missings)
    df2.rename_axis('Country',axis = 'columns',inplace = True)
    tot_players_list = pd.Series(tot_players_list,index = countries_with_missings)
    df2 = pd.concat([df2,tot_players_list],axis = 1)
    df2.columns = ['Total Players']
        
    return df2
    

In [9]:
def percentage_of_missings(df):
    count_missing = count_missings(df)
    count_player = count_players(df)
    df2 = pd.concat([count_missing,count_player],axis = 'columns')
    df2['% Missing'] = np.round((df2['Players without data']/df2['Total Players']) *100,2)
    return df2
    
    
    

In [10]:
def organize_to_multiindex(df,worldcup_name):
    df = df.T.reset_index().reset_index()
    df.iloc[:,0] = worldcup_name
    df.columns = range(df.shape[1])
    return df

In [11]:
worldcups = ['1930 Uruguay', '1934 Italy', '1938 France', '1950 Brazil', '1954 Switzerland', '1958 Sweden', '1962 Chile',
 '1966 England', '1970 Mexico', '1974 Germany', '1978 Argentina', '1982 Spain', '1986 Mexico', '1990 Italy', '1994 USA',
 '1998 France', '2002 Japan', '2006 Germany', '2010 South Africa', '2014 Brazil', '2018 Russia', '2022 Qatar']

In [14]:
path = '../3 - WorldCup DataFrames/worldcupfootball/{}.csv'

### Goal 1: Create a dataframe with the name of all players with missing data from all worldcups

Forming a initial dataframe to concat with others:

In [16]:
df1 = pd.read_csv(path.format(worldcups[0]),header = [0,1])
df1 = df1.iloc[:,1:]
df1.head()

Argentina                                       Belgium  \
                Name    Position Day Month  Year              Name   
0       Ángel Bossio  Goalkeeper   5     5  1905     Arnold Badjou   
1       Juan Botasso  Goalkeeper  23    10  1908        Jan De Bie   
2  Alberto Chividini    Defender  23     2  1907    Henri De Deken   
3   José Della Torre    Defender  23     3  1906    Nico Hoydonckx   
4      Juan Evaristo    Defender  20     6  1902  Théodore Nouwens   

                               ...               USA                        \
     Position Day Month  Year  ...              Name    Position Day Month   
0  Goalkeeper  25     6  1909  ...     Jimmy Douglas  Goalkeeper  12     1   
1  Goalkeeper   9     5  1892  ...      Frank Vaughn  Goalkeeper  18     2   
2    Defender   3     8  1907  ...  George Moorhouse    Defender   4     5   
3    Defender  29    12  1900  ...     Arnold Oliver    Defender  22     5   
4    Defender  17     2  1908  ...         Alec Wood    Defender  12     6   

                 Yugoslavia                              
   Year                Name    Position Day Month  Year  
0  1898      Milovan Jakšić  Goalkeeper  21     9  1909  
1  1902    Milan Stojanovic  Goalkeeper  28    12  1911  
2  1901     Milutin Ivković    Defender   3     3  1906  
3  1907  Branislav Markovic    Defender   0     0     0  
4  1907   Dragan Mihajlović    Defender  13    12  1906  

[5 rows x 65 columns]

In [17]:
missings_df1 = missings_by_team(df1)
missings_df1 = organize_to_multiindex(missings_df1,worldcups[0])
missings_df1

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,1930 Uruguay,Bolivia,José Chavarría,Luis Reyes Penaranda,Juan Argote,Miguel Brito,Diógenes Lara,Constantino Noya,Mario Alborta,José Bustamante,René Fernández,Rafael Mendez,Eduardo Reyes,0,0,0,0,0,0
1,1930 Uruguay,Mexico,Isidoro Sota,Francisco Gutiérrez,Alfredo Sanchez,Jesus Castro,Roberto Gayon,Felipe Olivares,Luis Perez,Jose Ruiz,0,0,0,0,0,0,0,0,0
2,1930 Uruguay,Paraguay,Modesto Denis,Eustacio Chamorro,Salvador Flores,Francisco Aguirre,Santiago Benitez,Eusébio Díaz,Romildo Etcheverry,Diego Florentin,Tranquilino Garcete,Saguier Carreras,Diogenes Dominguez,Lino Nessi,Amadeo Ortega,Bernabe Rivera,Gerardo Romero,Luis Vargas Peña,Jacinto Villalba
3,1930 Uruguay,Peru,Alberto Soria,Alberto Denegri,Jose Flores,Domingo Garcia,Julio Quintana,Juan Alfonso Valle,Jorge Gongora,Demetrio Neyra,Pablo Pacheco,0,0,0,0,0,0,0,0
4,1930 Uruguay,Romania,Andrei Glanzmann,Constantin Stanciu,Ilie Subaseanu,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,1930 Uruguay,USA,Bill O'Brian,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,1930 Uruguay,Yugoslavia,Branislav Markovic,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Now we concat the players with missing data from all worldcups in one dataframe:

In [18]:
for worldcup in worldcups[1:]:
    df = pd.read_csv(path.format(worldcup),header = [0,1])
    df = df.iloc[:,1:]
    if(len(players_without_data(df)) == 1): #If there is no missing data continue
        continue
    else:
        missings_df = missings_by_team(df)
        missings_df = organize_to_multiindex(missings_df,worldcup)
        missings_df1 = pd.concat([missings_df1,missings_df])

missings_df1.fillna(0,inplace =True)


In [19]:
missings_df1.head(10)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,1930 Uruguay,Bolivia,José Chavarría,Luis Reyes Penaranda,Juan Argote,Miguel Brito,Diógenes Lara,Constantino Noya,Mario Alborta,José Bustamante,René Fernández,Rafael Mendez,Eduardo Reyes,0,0,0,0,0,0
1,1930 Uruguay,Mexico,Isidoro Sota,Francisco Gutiérrez,Alfredo Sanchez,Jesus Castro,Roberto Gayon,Felipe Olivares,Luis Perez,Jose Ruiz,0,0,0,0,0,0,0,0,0
2,1930 Uruguay,Paraguay,Modesto Denis,Eustacio Chamorro,Salvador Flores,Francisco Aguirre,Santiago Benitez,Eusébio Díaz,Romildo Etcheverry,Diego Florentin,Tranquilino Garcete,Saguier Carreras,Diogenes Dominguez,Lino Nessi,Amadeo Ortega,Bernabe Rivera,Gerardo Romero,Luis Vargas Peña,Jacinto Villalba
3,1930 Uruguay,Peru,Alberto Soria,Alberto Denegri,Jose Flores,Domingo Garcia,Julio Quintana,Juan Alfonso Valle,Jorge Gongora,Demetrio Neyra,Pablo Pacheco,0,0,0,0,0,0,0,0
4,1930 Uruguay,Romania,Andrei Glanzmann,Constantin Stanciu,Ilie Subaseanu,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,1930 Uruguay,USA,Bill O'Brian,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,1930 Uruguay,Yugoslavia,Branislav Markovic,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,1934 Italy,Argentina,Ramon Astudillo,Enrique Chimento,José Nehin,Federico Wilde,Alfonso Lorenzo,Francisco Perez,0,0,0,0,0,0,0,0,0,0,0
1,1934 Italy,Egypt,Abdel Aziz,Ibrahim Abdelhamid,Sharli Hamidu,Ismail Rafaat,Mohammed Farid,Hassan Raghab,0,0,0,0,0,0,0,0,0,0,0
2,1934 Italy,Germany,Eugen Dienert,Sepp Streb,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [20]:
def multiindex(df):
    df.index = pd.MultiIndex.from_frame(df.iloc[:,:2])
    df = df.T
    df = df.iloc[2:,:]
    return df

In [21]:
missings = multiindex(missings_df1)
missings.to_csv('Missings.csv')
missings.to_excel('Missings.xlsx')

In [22]:
missings.head(5)

0          1930 Uruguay                                          \
1               Bolivia               Mexico           Paraguay   
2        José Chavarría         Isidoro Sota      Modesto Denis   
3  Luis Reyes Penaranda  Francisco Gutiérrez  Eustacio Chamorro   
4           Juan Argote      Alfredo Sanchez    Salvador Flores   
5          Miguel Brito         Jesus Castro  Francisco Aguirre   
6         Diógenes Lara        Roberto Gayon   Santiago Benitez   

0                                                                         \
1             Peru             Romania           USA          Yugoslavia   
2    Alberto Soria    Andrei Glanzmann  Bill O'Brian  Branislav Markovic   
3  Alberto Denegri  Constantin Stanciu             0                   0   
4      Jose Flores      Ilie Subaseanu             0                   0   
5   Domingo Garcia                   0             0                   0   
6   Julio Quintana                   0             0                   0   

0        1934 Italy                                     ...   1958 Sweden  \
1         Argentina               Egypt        Germany  ...       Hungary   
2   Ramon Astudillo          Abdel Aziz  Eugen Dienert  ...  Mihály Vasas   
3  Enrique Chimento  Ibrahim Abdelhamid     Sepp Streb  ...             0   
4        José Nehin       Sharli Hamidu              0  ...             0   
5    Federico Wilde       Ismail Rafaat              0  ...             0   
6   Alfonso Lorenzo      Mohammed Farid              0  ...             0   

0                          1962 Chile                               \
1           Paraguay         Colombia           CSSR        Mexico   
2    Agustin Miranda     Achito Vivas  Vladimir Kois  Alberto Baez   
3    Claudio Lezcano    Ignacio Calle              0             0   
4         Luis Silva  Eusebio Escobar              0             0   
5  Salvador Villalba         Luis Paz              0             0   
6                  0                0              0             0   

0                1970 Mexico                                        \
1 Switzerland    El Salvador                Morocco           Peru   
2  Fritz Kehl  David Cabrera  Ahmed Abdel Ouriaghly  Ruben Correra   
3           0   Tomas Pineda      Boujemaa Benkhrif              0   
4           0              0            Hamid Dahan              0   
5           0              0          Jilali Fadili              0   
6           0              0            Said Ghandi              0   

0         1982 Spain  
1             Kuwait  
2  Ahmad Marjan Adam  
3   Abdullah Ma'Yoof  
4     Mohammed Ahmad  
5   Nasser Al-Ghanem  
6    Mubarak Al-Issa  

[5 rows x 43 columns]

### Goal 2 Create a dataframe with the frequence and the percentage of missing data from each worldcup

In [23]:
perc_missings1 = percentage_of_missings(df1)
perc_missings1

C:\Users\italo\AppData\Local\Temp\ipykernel_21744\4079779035.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_players = (players).value_counts()[0]


,Players without data,Total Players,% Missing
Bolivia,11,17,64.71
Mexico,8,17,47.06
Paraguay,17,22,77.27
Peru,9,23,39.13
Romania,3,20,15.00
USA,1,18,5.56
Yugoslavia,1,17,5.88


In [24]:
perc_missings1 = perc_missings1.reset_index()
perc_missings1.columns = ['Teams','Players without data', 'Total Players', '% Missing']
perc_missings1 = organize_to_multiindex(perc_missings1,worldcups[0])

for worldcup in worldcups[1:]:
    df = pd.read_csv(path.format(worldcup),header = [0,1])
    df = df.iloc[:,1:]
    if(len(players_without_data(df)) == 1): #If there is no missing data continue
        continue
    else:
        perc_missings = percentage_of_missings(df)
        perc_missings = perc_missings.reset_index()
        perc_missings.columns = ['Teams','Players without data', 'Total Players', '% Missing']
        perc_missings = organize_to_multiindex(perc_missings,worldcup)
        perc_missings1 = pd.concat([perc_missings1,perc_missings])

perc_missings1.fillna(0,inplace =True)
percentage = multiindex(perc_missings1)
percentage


C:\Users\italo\AppData\Local\Temp\ipykernel_21744\4079779035.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_players = (players).value_counts()[0]
C:\Users\italo\AppData\Local\Temp\ipykernel_21744\4079779035.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_players = (players).value_counts()[0]
C:\Users\italo\AppData\Local\Temp\ipykernel_21744\4079779035.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  total_

0  1930 Uruguay                                                1934 Italy  \
1         Teams Players without data Total Players % Missing        Teams   
2       Bolivia                   11            17     64.71    Argentina   
3        Mexico                    8            17     47.06        Egypt   
4      Paraguay                   17            22     77.27      Germany   
5          Peru                    9            23     39.13      Hungary   
6       Romania                    3            20      15.0      Romania   
7           USA                    1            18      5.56  Switzerland   
8    Yugoslavia                    1            17      5.88          USA   
9             0                    0             0         0            0   
10            0                    0             0         0            0   

0                                                      1938 France  \
1  Players without data Total Players % Missing              Teams   
2                     6            18     33.33               CSSR   
3                     6            16      37.5               Cuba   
4                     2            22      9.09             France   
5                     5            22     22.73            Hungary   
6                     2            22      9.09        Netherlands   
7                     4            22     18.18  Netherlands India   
8                     4            19     21.05             Poland   
9                     0             0         0            Romania   
10                    0             0         0        Switzerland   

0                        ...    1962 Chile            1970 Mexico  \
1  Players without data  ... Total Players % Missing        Teams   
2                     2  ...            22      4.55  El Salvador   
3                     8  ...            22     18.18      Morocco   
4                     1  ...            22      4.55         Peru   
5                     4  ...            22      4.55            0   
6                     5  ...             0         0            0   
7                    14  ...             0         0            0   
8                     2  ...             0         0            0   
9                     1  ...             0         0            0   
10                    3  ...             0         0            0   

0                                               1982 Spain  \
1  Players without data Total Players % Missing      Teams   
2                     2            22      9.09     Kuwait   
3                    10            19     52.63          0   
4                     1            22      4.55          0   
5                     0             0         0          0   
6                     0             0         0          0   
7                     0             0         0          0   
8                     0             0         0          0   
9                     0             0         0          0   
10                    0             0         0          0   

0                                                
1  Players without data Total Players % Missing  
2                     7            22     31.82  
3                     0             0         0  
4                     0             0         0  
5                     0             0         0  
6                     0             0         0  
7                     0             0         0  
8                     0             0         0  
9                     0             0         0  
10                    0             0         0  

[9 rows x 36 columns]

In [25]:
missing_teams = ['1930 Uruguay', '1934 Italy', '1938 France', '1950 Brazil', '1954 Switzerland', '1958 Sweden', '1962 Chile',
 '1970 Mexico', '1982 Spain']

In [26]:
Total_missings = []
for team in missing_teams:
    Total_missings.append(percentage[team]['Players without data'])

total_missings = np.array(Total_missings).sum()

In [27]:
Total_players = []
for team in missing_teams:
    Total_players.append(percentage[team]['Total Players'])

total_players = np.array(Total_players).sum()

In [28]:
print(f'Between the teams with missing data, the percentage of players without birthdate is {total_missings*100/total_players:.2f}%')

Between the teams with missing data, the percentage of players without birthdate is 20.91%


In [29]:
percentage.to_excel('Percentage.xlsx')

In [30]:
percentage.to_csv('Percentage.csv')